# Notebook_02_Furusato_Apply_Ontology_Metadata

**Workshop version:** 2.7.0

This standalone Fabric PySpark Notebook bulk-registers semantic metadata for
the canonical Furusato Ontology: 10 Entity types, 72 static Properties,
1 time-series Property, and 15 Relationship types.

It uses the supported Fabric Ontology REST `getDefinition` /
`updateDefinition` flow instead of an MCP write path. The current definition
is fetched first, all IDs, bindings, contextualizations, and unrelated parts
are preserved, and only `semanticEnrichment` is changed. The default run is
preview-only.

In [ ]:
# Fabric parameter cell
PARTICIPANT_ID = "001"
ONTOLOGY_DISPLAY_NAME = f"ONT_Furusato_{PARTICIPANT_ID}"
EXPECTED_WORKSPACE_NAME = ""

APPLY_CHANGES = False
APPLY_CONFIRMATION = ""
EXCLUSIVE_APPLY_WINDOW_CONFIRMED = False

OPERATION_TIMEOUT_SECONDS = 600

## Canonical name-based metadata manifest

The manifest contains no workspace, item, Entity, Property, or Relationship
IDs. The Notebook resolves the current item and definition parts by exact
display name and stable schema names at runtime.

In [ ]:
import json

ONTOLOGY_METADATA = json.loads(r'''{
  "schemaVersion": "1.0",
  "packageVersion": "2.7.0",
  "ontologyDisplayNameTemplate": "ONT_Furusato_<PID>",
  "description": "Canonical semantic enrichment for the Furusato Fabric Workshop Ontology. IDs and data bindings are resolved from the current item definition and are never stored here.",
  "expectedContract": {
    "entityTypes": 10,
    "staticProperties": 72,
    "timeseriesProperties": 1,
    "relationshipTypes": 15
  },
  "entities": {
    "Donation": {
      "semanticEnrichment": {
        "synonyms": [
          "donation",
          "donations",
          "contribution",
          "contributions",
          "donation record",
          "contribution record",
          "donation transaction",
          "donation order",
          "completed donation",
          "hometown tax donation",
          "Furusato Nozei donation",
          "Furusato tax contribution",
          "寄付",
          "寄附",
          "寄付申込",
          "寄附申込",
          "ふるさと納税"
        ],
        "description": "Synthetic static baseline hometown-tax Donation row (ふるさと納税の寄付／寄附) and the only static fact grain for transaction counts and JPY amounts. It is an independent snapshot, not an approved, promoted, deduplicated, or finalized form of an Eventhouse observation. Each entity links to exactly one Donor, one recipient Municipality, and one selected Gift option in this dataset.",
        "customAttributes": {
          "businessRole": "transaction fact",
          "grain": "one static baseline donation",
          "sensitivity": "Synthetic-Transaction",
          "currency": "JPY",
          "factPolicy": "only source for transaction counts and donation amounts",
          "timeSeriesPolicy": "Municipality time-series observations are excluded"
        }
      },
      "properties": {
        "DonationId": {
          "valueType": "BigInt",
          "isKey": true,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stable numeric identifier for one static Donation transaction. When a user says 寄付ID or Donation ID followed by a number, filter this property for exact identity; never reinterpret that number as DonationAmountYen. It is the transaction key and must not be used as a grouping dimension for an overall aggregate.",
            "customAttributes": {
              "businessRole": "stable transaction identifier",
              "grain": "one static Donation",
              "format": "integer ID",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationDisplayName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Display-only label in the format Donation followed by DonationId. Use DonationId for identity.",
            "customAttributes": {
              "businessRole": "display label",
              "grain": "one value per Donation",
              "format": "Donation {DonationId}",
              "language": "en",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationAmountYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Completed donation amount (寄付金額／寄附金額／納税額) in whole Japanese yen for exactly one static Donation. SUM this property for donation totals and never sum IDs or Municipality time-series observations into the same result.",
            "customAttributes": {
              "businessRole": "additive fact measure",
              "grain": "one value per Donation",
              "unit": "JPY",
              "format": "whole-yen integer",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "SUM"
            }
          }
        },
        "DonatedAtUtc": {
          "valueType": "DateTime",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Canonical UTC timestamp of the static baseline Donation. Use it for exact sorting, filtering, minimum, maximum, and deterministic latest-record selection.",
            "customAttributes": {
              "businessRole": "canonical transaction timestamp",
              "grain": "one value per Donation",
              "timezone": "UTC",
              "format": "DateTime",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "MIN or MAX only"
            }
          }
        },
        "DonatedAtJstText": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Display-only representation of DonatedAtUtc in Japan Standard Time (日本時間), formatted as yyyy-MM-dd HH:mm:ss followed by the literal suffix JST. Do not use this string for chronological sorting when DonatedAtUtc is available.",
            "customAttributes": {
              "businessRole": "localized display timestamp",
              "grain": "one value per Donation",
              "timezone": "Asia/Tokyo (+09:00)",
              "format": "yyyy-MM-dd HH:mm:ss JST",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationDateJst": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Japan Standard Time calendar date of the Donation (寄付日／寄附日). Use it for local-day filtering and grouping, not for exact timestamp ordering.",
            "customAttributes": {
              "businessRole": "local calendar date",
              "grain": "one value per Donation",
              "timezone": "Asia/Tokyo",
              "format": "yyyy-MM-dd",
              "unit": "day",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationYearMonthJst": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Japan Standard Time calendar year and month of the Donation (寄付年月／寄附年月). Use it for local-month filtering and grouping. Normalize an unambiguous Japanese two-digit Gregorian year such as 25年 to 2025 before matching the stored yyyy-MM value.",
            "customAttributes": {
              "businessRole": "local calendar month",
              "grain": "one value per Donation",
              "timezone": "Asia/Tokyo",
              "format": "yyyy-MM",
              "unit": "month",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationYearMonthJaShort": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Japanese short Gregorian year-month label for the Donation, formatted yyyy年M月, such as 2025年1月. Use it only when the user supplies that short Japanese month wording; DonationYearMonthJst remains the canonical yyyy-MM calendar key.",
            "customAttributes": {
              "businessRole": "Japanese short local calendar month",
              "grain": "one value per Donation",
              "timezone": "Asia/Tokyo",
              "format": "yyyy年M月",
              "unit": "month",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationPaymentMethod": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stored payment method, payment channel, checkout channel, or tender label (支払方法／決済方法／決済手段) for the completed Donation. It is a controlled Japanese-or-brand label and must not be interpreted as a Supplier.",
            "customAttributes": {
              "businessRole": "Japanese or brand categorical attribute",
              "grain": "one value per Donation",
              "language": "ja-JP and brand names",
              "format": "controlled text, 4 values",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationPaymentMethodEn": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Canonical English payment-method label for DonationPaymentMethod. Use it for English filtering and grouping; return both labels when the user asks for a translation.",
            "customAttributes": {
              "businessRole": "canonical English categorical attribute",
              "grain": "one value per Donation",
              "language": "en",
              "format": "controlled text, 4 values",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "none"
            }
          }
        },
        "DonationDataLayer": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Canonical layer label for this detail entity. Its only value is StaticSyntheticSnapshot, which is not a payment method or a time-series event type. The constant is a deliberate exception to the rule that a single-valued column carries no information: it is the declared boundary marker that keeps the static snapshot separable from the operational observation layer, so a query can never silently blend the two. Never filter it away, never group by it, and never present it as a lifecycle or approval state.",
            "customAttributes": {
              "businessRole": "data layer discriminator",
              "grain": "one Donation",
              "canonicalValue": "StaticSyntheticSnapshot",
              "paymentMethodRole": "none",
              "defaultAggregation": "none",
              "constantByDesign": "true",
              "boundaryRole": "static-versus-observation layer marker",
              "modelingException": "a single-valued technical property is retained on purpose so the static layer is explicit in every projection"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "Donor": {
      "semanticEnrichment": {
        "synonyms": [
          "donor",
          "donors",
          "contributor",
          "contributors",
          "supporter",
          "supporters",
          "hometown tax donor",
          "taxpayer",
          "寄付者",
          "寄附者",
          "支援者",
          "納税者"
        ],
        "description": "Synthetic workshop donor (寄付者／寄附者) who may make zero or more Donations. Residence is defined only by DonorLivesInPrefecture and must never be interpreted as recipient geography.",
        "customAttributes": {
          "businessRole": "actor dimension",
          "grain": "one synthetic donor",
          "sensitivity": "Synthetic-Personal",
          "dataNature": "synthetic, no real PII",
          "geographyRole": "donor residence only"
        }
      },
      "properties": {
        "DonorId": {
          "valueType": "BigInt",
          "isKey": true,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stable numeric identifier for one synthetic Donor entity. Names aren't guaranteed unique, so use this ID to disambiguate people.",
            "customAttributes": {
              "businessRole": "stable entity identifier",
              "grain": "one Donor",
              "format": "integer ID",
              "sensitivity": "Synthetic workshop data",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Synthetic Japanese personal-style name of the donor (寄付者名／支援者名). It is not guaranteed to be unique and does not represent a real person.",
            "customAttributes": {
              "businessRole": "Japanese label",
              "grain": "one value per Donor",
              "language": "ja-JP",
              "format": "synthetic Japanese name",
              "sensitivity": "Synthetic-Personal",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorDisplayName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Display-only composite in the format {DonorName} / Donor {DonorId}. Use DonorId for identity and this property for human-readable output.",
            "customAttributes": {
              "businessRole": "display label",
              "grain": "one value per Donor",
              "format": "{DonorName} / Donor {DonorId}",
              "language": "ja-JP",
              "sensitivity": "Synthetic-Personal",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorAge": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Whole-year age of the synthetic donor (年齢) in the static donor record. It is not derived from the Donation timestamp and must not be summed.",
            "customAttributes": {
              "businessRole": "demographic attribute",
              "grain": "one value per Donor",
              "unit": "years",
              "format": "integer",
              "validRange": "20-79",
              "sensitivity": "Synthetic-Personal",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorOccupation": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Japanese controlled occupation or job label (職業／職種／仕事) for the synthetic donor. It describes the donor's job category, not employer, recipient, or Supplier type.",
            "customAttributes": {
              "businessRole": "Japanese categorical attribute",
              "grain": "one value per Donor",
              "language": "ja-JP",
              "format": "controlled text, 28 values",
              "sensitivity": "Synthetic-Personal",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorOccupationEn": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Canonical English occupation label for DonorOccupation. Use it for English filtering and grouping; preserve DonorOccupation when returning the stored Japanese label.",
            "customAttributes": {
              "businessRole": "canonical English categorical attribute",
              "grain": "one value per Donor",
              "language": "en",
              "format": "controlled text, 28 values",
              "sensitivity": "Synthetic-Personal",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated number of static Donations made by this Donor.",
            "customAttributes": {
              "businessRole": "donor snapshot metric",
              "grain": "one Donor",
              "measure": "count of Donation",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorStaticTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated cumulative static Donation amount made by this Donor in JPY.",
            "customAttributes": {
              "businessRole": "donor snapshot metric",
              "grain": "one Donor",
              "unit": "JPY",
              "metric": "cumulative amount",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorStaticMaxYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Maximum single static Donation amount made by this Donor in JPY; it is not the cumulative amount.",
            "customAttributes": {
              "businessRole": "donor snapshot metric",
              "grain": "one Donor",
              "unit": "JPY",
              "metric": "single-donation maximum",
              "defaultAggregation": "none"
            }
          }
        },
        "DonorOverallAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Cumulative static donation-amount rank of this Donor nationwide. Deterministic row-number rank (row_number) over DonorStaticTotalYen desc, DonorId asc: equal amounts never share a rank because DonorId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "cumulative amount rank",
              "grain": "one Donor",
              "rankScope": "all Donors nationwide",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "DonorStaticTotalYen desc, DonorId asc",
              "rankTieBreak": "DonorId ascending",
              "rankPartition": "none (single global window)"
            }
          }
        },
        "DonorResidenceAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Cumulative static donation-amount rank of this Donor inside the residence Prefecture. Deterministic row-number rank (row_number) over DonorStaticTotalYen desc, DonorId asc within each PrefectureId: equal amounts never share a rank because DonorId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "cumulative amount rank",
              "grain": "one Donor",
              "rankScope": "within residence Prefecture",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "DonorStaticTotalYen desc, DonorId asc",
              "rankTieBreak": "DonorId ascending",
              "rankPartition": "PrefectureId"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "Gift": {
      "semanticEnrichment": {
        "synonyms": [
          "gift",
          "return gift",
          "return gifts",
          "thank-you gift",
          "reward",
          "benefit",
          "perk",
          "gift option",
          "hometown tax gift",
          "返礼品",
          "お礼品",
          "商品",
          "特典",
          "返礼品オプション"
        ],
        "description": "Synthetic return-gift option selected by a Donation (返礼品／お礼品／商品). It is a catalog dimension, not the donated money and not a transaction fact. If a Municipality is named, resolve that Municipality and traverse MunicipalityCatalogsGift before applying a GiftName CONTAINS candidate filter. Resolve exactly one stable GiftId before Supplier or Donation traversal.",
        "customAttributes": {
          "businessRole": "catalog item dimension",
          "grain": "one return-gift option",
          "sensitivity": "Public-Synthetic-Catalog",
          "monetaryRole": "none",
          "recipientPath": "DonationToMunicipality only"
        }
      },
      "properties": {
        "GiftId": {
          "valueType": "BigInt",
          "isKey": true,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stable numeric identifier for one Gift catalog entity. Use it to connect Donations, catalog Municipalities, categories, and registered Suppliers.",
            "customAttributes": {
              "businessRole": "stable entity identifier",
              "grain": "one Gift",
              "format": "integer ID",
              "defaultAggregation": "none"
            }
          }
        },
        "GiftName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Core Japanese product or service name of the return-gift option (返礼品名／お礼品名／商品名). It usually opens with a place-of-origin word such as a prefecture name, which is display text and not authoritative geography. For a Municipality-qualified colloquial label, traverse MunicipalityCatalogsGift first, filter this property with CONTAINS on the core noun, and verify the remaining numeric and descriptive tokens. Core names can repeat, so resolve one stable GiftId before further traversal.",
            "customAttributes": {
              "businessRole": "Japanese label",
              "grain": "one value per Gift",
              "language": "ja-JP",
              "format": "Japanese text",
              "sensitivity": "Public-Synthetic-Catalog",
              "defaultAggregation": "none"
            }
          }
        },
        "GiftDisplayName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Display-only composite in the format {GiftName} / Gift {GiftId}. Use it for unambiguous human-readable output and GiftId as the stable identity.",
            "customAttributes": {
              "businessRole": "display label",
              "grain": "one value per Gift",
              "format": "{GiftName} / Gift {GiftId}",
              "language": "ja-JP",
              "sensitivity": "Public-Synthetic-Catalog",
              "defaultAggregation": "none"
            }
          }
        },
        "GiftSearchTerms": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pipe-delimited search-only text for resolving shortened or colloquial Gift wording. It contains the core GiftName, the English GiftNameEn, the Japanese CategoryName, and the English CategoryNameEn. Place-of-origin words often appear inside those name strings, but they are display text and not authoritative geography, so resolve catalog and recipient geography through MunicipalityCatalogsGift and DonationToMunicipality. Filter it with a case-insensitive substring match for every distinctive noun supplied by the user; never display, group, count, or treat it as Gift identity.",
            "customAttributes": {
              "businessRole": "search-only Gift resolver",
              "grain": "one value per Gift",
              "language": "ja-JP,en",
              "format": "{GiftName} | {GiftNameEn} | {CategoryName} | {CategoryNameEn}",
              "sensitivity": "Public-Synthetic-Catalog",
              "queryPattern": "FILTER lower(g.GiftSearchTerms) CONTAINS lower('<distinctive token>') for every supplied token",
              "displayPolicy": "never display, group, or aggregate",
              "defaultAggregation": "none"
            }
          }
        },
        "GiftNotes": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Japanese free-text note, notes, remarks, or description for the return-gift option (備考／注意事項／説明). Notes are descriptive only and do not define Supplier, recipient geography, category, or monetary amount.",
            "customAttributes": {
              "businessRole": "descriptive text",
              "grain": "one value per Gift",
              "language": "ja-JP",
              "format": "free text",
              "sensitivity": "Public-Synthetic-Catalog",
              "defaultAggregation": "none"
            }
          }
        },
        "GiftStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Authoritative pre-aggregated count of static Donations that selected this Gift. After resolving one GiftId, use this property directly for an all-time Gift count; do not join Suppliers or recount Donation detail.",
            "customAttributes": {
              "businessRole": "gift snapshot metric",
              "grain": "one Gift",
              "measure": "count of Donation",
              "queryPolicy": "select directly from the resolved Gift without traversing any relationship",
              "defaultAggregation": "none"
            }
          }
        },
        "GiftStaticTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Authoritative pre-aggregated JPY amount of static Donations that selected this Gift. After resolving one GiftId, use this property directly for an all-time Gift total; do not join Suppliers or use SUM(DISTINCT DonationAmountYen).",
            "customAttributes": {
              "businessRole": "gift snapshot metric",
              "grain": "one Gift",
              "unit": "JPY",
              "queryPolicy": "select directly from the resolved Gift without traversing any relationship",
              "defaultAggregation": "none"
            }
          }
        },
        "GiftAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Static donation-amount rank of this Gift across all Gifts. Deterministic row-number rank (row_number) over GiftStaticTotalYen desc, GiftId asc: equal amounts never share a rank because GiftId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "gift amount rank",
              "grain": "one Gift",
              "rankScope": "all Gifts",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "GiftStaticTotalYen desc, GiftId asc",
              "rankTieBreak": "GiftId ascending",
              "rankPartition": "none (single global window)"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "GiftCategory": {
      "semanticEnrichment": {
        "synonyms": [
          "gift category",
          "return-gift category",
          "reward category",
          "benefit category",
          "product category",
          "genre",
          "返礼品カテゴリ",
          "返礼品分類",
          "お礼品カテゴリ",
          "特典カテゴリ",
          "ジャンル"
        ],
        "description": "Controlled classification of return-gift options (返礼品カテゴリ). Category counts and amounts must be aggregated from Donation through DonationSelectedGift and GiftInCategory; the category itself is not a transaction fact.",
        "customAttributes": {
          "businessRole": "catalog classification dimension",
          "grain": "one controlled return-gift category",
          "sensitivity": "Public-Synthetic-Catalog",
          "metricPolicy": "aggregate at Donation grain"
        }
      },
      "properties": {
        "CategoryId": {
          "valueType": "BigInt",
          "isKey": true,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stable numeric identifier for one GiftCategory entity. Use it with the stored category labels and never sum or average it.",
            "customAttributes": {
              "businessRole": "stable entity identifier",
              "grain": "one GiftCategory",
              "format": "integer ID",
              "defaultAggregation": "none"
            }
          }
        },
        "CategoryName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Canonical Japanese return-gift category label (返礼品カテゴリ／返礼品分類). Category metrics must be calculated from Donation, not from this label.",
            "customAttributes": {
              "businessRole": "Japanese controlled label",
              "grain": "one value per GiftCategory",
              "language": "ja-JP",
              "format": "controlled text, 30 values",
              "sensitivity": "Public-Synthetic-Catalog",
              "defaultAggregation": "none"
            }
          }
        },
        "CategoryNameEn": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Canonical English return-gift category label paired one-to-one with CategoryName. Use it for English filtering, grouping, and display; CategoryId remains the identity.",
            "customAttributes": {
              "businessRole": "canonical English label",
              "grain": "one value per GiftCategory",
              "language": "en",
              "format": "controlled text, 30 values",
              "sensitivity": "Public-Synthetic-Catalog",
              "defaultAggregation": "none"
            }
          }
        },
        "CategorySearchTerms": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pipe-delimited alias text holding the Japanese CategoryName, the English CategoryNameEn, and the literal marker gift category. Use it only to resolve colloquial return-gift category wording such as 海鮮系, 魚介, or seafood with a case-insensitive substring match. If CategoryName does not match, query this alias index before claiming no category data. Never display, group, count, or treat this property as the category identity.",
            "customAttributes": {
              "businessRole": "search-only alias index",
              "grain": "one value per GiftCategory",
              "language": "ja-JP,en",
              "format": "{CategoryName} | {CategoryNameEn} | gift category",
              "sensitivity": "Public-Synthetic-Catalog",
              "queryPattern": "FILTER lower(c.CategorySearchTerms) CONTAINS lower('<alias>')",
              "displayPolicy": "never display, group, or aggregate"
            }
          }
        },
        "CategoryStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated static Donation count for this GiftCategory across the full snapshot.",
            "customAttributes": {
              "businessRole": "category snapshot metric",
              "grain": "one GiftCategory",
              "measure": "count of Donation",
              "defaultAggregation": "none"
            }
          }
        },
        "CategoryStaticTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated static Donation amount for this GiftCategory across the full snapshot.",
            "customAttributes": {
              "businessRole": "category snapshot metric",
              "grain": "one GiftCategory",
              "unit": "JPY",
              "defaultAggregation": "none"
            }
          }
        },
        "CategoryAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Static donation-amount rank of this GiftCategory across all GiftCategories. Deterministic row-number rank (row_number) over CategoryStaticTotalYen desc, CategoryId asc: equal amounts never share a rank because CategoryId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "category amount rank",
              "grain": "one GiftCategory",
              "rankScope": "all GiftCategories",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "CategoryStaticTotalYen desc, CategoryId asc",
              "rankTieBreak": "CategoryId ascending",
              "rankPartition": "none (single global window)"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "Municipality": {
      "semanticEnrichment": {
        "synonyms": [
          "municipality",
          "municipalities",
          "recipient municipality",
          "donation destination",
          "local government",
          "local authority",
          "city",
          "town",
          "village",
          "ward",
          "自治体",
          "市区町村",
          "受入自治体",
          "寄付先自治体",
          "寄附先自治体"
        ],
        "description": "Japanese municipality (自治体／市区町村) that may receive zero or more Donations. It also keys operational time-series observations; those observations are not Donation entities and have no Donor, Gift, or Supplier identity.",
        "customAttributes": {
          "businessRole": "recipient geography dimension",
          "grain": "one entity per Japanese municipality",
          "sensitivity": "Public",
          "timeSeriesRole": "context key only",
          "factPolicy": "Donation is the transaction fact"
        }
      },
      "properties": {
        "MunicipalityId": {
          "valueType": "String",
          "isKey": true,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stable identifier for one Municipality entity. Preserve leading zeros if present and use it as the relationship key; do not aggregate it.",
            "customAttributes": {
              "businessRole": "stable entity identifier",
              "grain": "one Municipality",
              "format": "string ID",
              "defaultAggregation": "none"
            }
          }
        },
        "MunicipalityName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Japanese municipality name (自治体名／市区町村名) without a prefecture qualifier. Names are not globally unique, so include MunicipalityId or MunicipalityDisplayName when ambiguity is possible.",
            "customAttributes": {
              "businessRole": "Japanese label",
              "grain": "one value per Municipality",
              "language": "ja-JP",
              "format": "Japanese text",
              "sensitivity": "Public",
              "defaultAggregation": "none"
            }
          }
        },
        "MunicipalityDisplayName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Display-only composite label in the format {MunicipalityName} / {PrefectureName}. It does not contain MunicipalityId, so use MunicipalityId as the identity and this property only for human-readable output.",
            "customAttributes": {
              "businessRole": "display label",
              "grain": "one value per Municipality",
              "format": "{MunicipalityName} / {PrefectureName}",
              "language": "ja-JP",
              "sensitivity": "Public",
              "defaultAggregation": "none"
            }
          }
        },
        "MunicipalityStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated count of static Donations received by this Municipality.",
            "customAttributes": {
              "businessRole": "received snapshot metric",
              "grain": "one Municipality",
              "measure": "count of Donation",
              "defaultAggregation": "none"
            }
          }
        },
        "MunicipalityStaticTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated static Donation amount received by this Municipality in JPY.",
            "customAttributes": {
              "businessRole": "received snapshot metric",
              "grain": "one Municipality",
              "unit": "JPY",
              "defaultAggregation": "none"
            }
          }
        },
        "MunicipalityAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Received-amount rank of this Municipality across all Municipalities. Deterministic row-number rank (row_number) over MunicipalityStaticTotalYen desc, MunicipalityId asc: equal amounts never share a rank because MunicipalityId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "received amount rank",
              "grain": "one Municipality",
              "rankScope": "all Municipalities",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "MunicipalityStaticTotalYen desc, MunicipalityId asc",
              "rankTieBreak": "MunicipalityId ascending",
              "rankPartition": "none (single global window)"
            }
          }
        },
        "MunicipalityPrefAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Received-amount rank of this Municipality inside its containing Prefecture. Deterministic row-number rank (row_number) over MunicipalityStaticTotalYen desc, MunicipalityId asc within each PrefectureId: equal amounts never share a rank because MunicipalityId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "received amount rank",
              "grain": "one Municipality",
              "rankScope": "within containing Prefecture",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "MunicipalityStaticTotalYen desc, MunicipalityId asc",
              "rankTieBreak": "MunicipalityId ascending",
              "rankPartition": "PrefectureId"
            }
          }
        }
      },
      "timeseriesProperties": {
        "IncomingDonationAmountYen": {
          "valueType": "BigInt",
          "semanticEnrichment": {
            "description": "Donation amount observed in an operational Municipality time-series event. It binds to the raw ingested DonationEvents rows through DonatedAt and DonationAmountYen, so its values are raw observations: deduplication = none, and duplicate EventIDs are retained. It is measured in JPY but is not a static Donation entity property; never use it to increase Donation entity counts or to identify a Donor, Gift, Supplier, or DonationId. The Eventhouse remains the numeric authority for operational observation metrics; this property is a modeled mirror of them.",
            "customAttributes": {
              "businessRole": "operational time-series measure",
              "grain": "one ingested event observation for one Municipality",
              "unit": "JPY",
              "timezone": "event timestamp DonatedAt is UTC",
              "sensitivity": "Synthetic-Transaction",
              "defaultAggregation": "SUM only within an explicitly requested time-series scope",
              "staticFactPolicy": "must not be combined with Donation entity counts or amounts",
              "deduplication": "none - raw ingested rows; duplicate EventIDs retained",
              "sourceTable": "DonationEvents (raw), not DonationObservationSummaryForAgent",
              "numericAuthority": "Eventhouse curated view; this property is a modeled mirror"
            }
          }
        }
      }
    },
    "MunicipalityCategoryMetric": {
      "semanticEnrichment": {
        "synonyms": [
          "municipality category metric",
          "municipality category summary",
          "自治体別カテゴリ指標",
          "自治体別返礼品カテゴリ実績"
        ],
        "description": "Pre-aggregated current-static-snapshot metric for one Municipality and one GiftCategory pair with at least one Donation. It excludes time-series observations and Suppliers; values must not be summed again outside this grain.",
        "customAttributes": {
          "businessRole": "semantic metric entity",
          "grain": "one Municipality x GiftCategory pair",
          "identityProperty": "MunCategoryMetricId",
          "keyType": "String",
          "snapshotScope": "current static Donation only",
          "supplierPolicy": "excluded",
          "defaultAggregation": "none"
        }
      },
      "properties": {
        "MunCategoryMetricId": {
          "valueType": "String",
          "isKey": true,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Synthetic stable identifier for one Municipality x GiftCategory snapshot metric row, encoded as {MunicipalityId}-{CategoryId:02}, for example 452025-01. The measure values are separate properties on this entity. The same NN-NN text can also occur as a PrefCategoryMetricId or a PrefDonationFlowId, so the entity type - not the string shape - disambiguates the key; always qualify it as a MunicipalityCategoryMetric identifier.",
            "customAttributes": {
              "businessRole": "composite metric-row identifier",
              "grain": "one Municipality x GiftCategory pair",
              "format": "string ID",
              "defaultAggregation": "none",
              "keyEncoding": "{MunicipalityId}-{CategoryId:02}",
              "keyExample": "452025-01",
              "keyAmbiguity": "the same NN-NN text may exist in another metric entity type; qualify by entity type"
            }
          }
        },
        "MunCategoryStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated static Donation count for one Municipality x GiftCategory pair.",
            "customAttributes": {
              "businessRole": "cross-dimensional snapshot metric",
              "grain": "one Municipality x GiftCategory pair",
              "measure": "count of Donation",
              "defaultAggregation": "none"
            }
          }
        },
        "MunCategoryTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated static Donation amount for one Municipality x GiftCategory pair.",
            "customAttributes": {
              "businessRole": "cross-dimensional snapshot metric",
              "grain": "one Municipality x GiftCategory pair",
              "unit": "JPY",
              "defaultAggregation": "none"
            }
          }
        },
        "MunCategoryAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Amount rank of this GiftCategory inside one Municipality. Deterministic row-number rank (row_number) over MunCategoryTotalYen desc, CategoryId asc within each MunicipalityId: equal amounts never share a rank because CategoryId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "cross-dimensional amount rank",
              "grain": "one Municipality x GiftCategory pair",
              "rankScope": "within Municipality",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "MunCategoryTotalYen desc, CategoryId asc",
              "rankTieBreak": "CategoryId ascending",
              "rankPartition": "MunicipalityId"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "Prefecture": {
      "semanticEnrichment": {
        "synonyms": [
          "prefecture",
          "prefectures",
          "Japanese prefecture",
          "recipient prefecture",
          "donor residence prefecture",
          "都道府県",
          "都府県"
        ],
        "description": "Japanese prefecture (都道府県) used in two separate geography paths: recipient geography through Municipality and donor-residence geography through Donor. It is a reference dimension, not a Donation fact grain.",
        "customAttributes": {
          "businessRole": "reference geography dimension",
          "grain": "one entity per Japanese prefecture",
          "sensitivity": "Public",
          "language": "ja-JP,en",
          "geographyRoles": "recipient prefecture and donor-residence prefecture"
        }
      },
      "properties": {
        "PrefectureId": {
          "valueType": "BigInt",
          "isKey": true,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stable numeric identifier for one Japanese Prefecture entity. Use it to disambiguate names and as the relationship key; do not aggregate it. If the user explicitly asks for a 都道府県 or Prefecture result, keep this grain and never substitute a Municipality.",
            "customAttributes": {
              "businessRole": "stable entity identifier",
              "grain": "one Prefecture",
              "format": "integer ID",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefectureName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Official Japanese display label of the prefecture (都道府県名). Preserve the exact stored Japanese text in output.",
            "customAttributes": {
              "businessRole": "Japanese label",
              "grain": "one value per Prefecture",
              "language": "ja-JP",
              "format": "Japanese text",
              "sensitivity": "Public",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefectureNameEn": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Canonical English label of the prefecture. Use it for English lookup and display; PrefectureId remains the identity.",
            "customAttributes": {
              "businessRole": "English label",
              "grain": "one value per Prefecture",
              "language": "en",
              "format": "English text",
              "sensitivity": "Public",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefReceivedStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated count of static Donations received by municipalities in this Prefecture.",
            "customAttributes": {
              "businessRole": "received snapshot metric",
              "grain": "one Prefecture",
              "measure": "count of Donation",
              "snapshotScope": "current static Donation only",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefReceivedTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated JPY amount received by municipalities in this Prefecture.",
            "customAttributes": {
              "businessRole": "received snapshot metric",
              "grain": "one Prefecture",
              "unit": "JPY",
              "snapshotScope": "current static Donation only",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefReceivedAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Received-amount rank of this Prefecture across all Prefectures. Deterministic row-number rank (row_number) over PrefReceivedTotalYen desc, PrefectureId asc: equal amounts never share a rank because PrefectureId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "received amount rank",
              "grain": "one Prefecture",
              "rankScope": "all Prefectures",
              "sort": "amount descending",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "PrefReceivedTotalYen desc, PrefectureId asc",
              "rankTieBreak": "PrefectureId ascending",
              "rankPartition": "none (single global window)"
            }
          }
        },
        "PrefResidentStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated count of static Donations made by Donors who live in this Prefecture.",
            "customAttributes": {
              "businessRole": "resident-origin snapshot metric",
              "grain": "one Prefecture",
              "measure": "count of Donation",
              "geographyRole": "donor residence",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefResidentTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated JPY amount donated by Donors who live in this Prefecture.",
            "customAttributes": {
              "businessRole": "resident-origin snapshot metric",
              "grain": "one Prefecture",
              "unit": "JPY",
              "geographyRole": "donor residence",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefResidentAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Resident-amount rank of this Prefecture across all Prefectures. Deterministic row-number rank (row_number) over PrefResidentTotalYen desc, PrefectureId asc: equal amounts never share a rank because PrefectureId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "resident-origin amount rank",
              "grain": "one Prefecture",
              "rankScope": "all Prefectures",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "PrefResidentTotalYen desc, PrefectureId asc",
              "rankTieBreak": "PrefectureId ascending",
              "rankPartition": "none (single global window)"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "PrefectureCategoryMetric": {
      "semanticEnrichment": {
        "synonyms": [
          "prefecture category metric",
          "prefecture category summary",
          "都道府県別カテゴリ指標",
          "都道府県別返礼品カテゴリ実績"
        ],
        "description": "Pre-aggregated current-static-snapshot metric for one recipient Prefecture and one GiftCategory pair with at least one Donation. It excludes time-series observations and Suppliers; values must not be summed again outside this grain.",
        "customAttributes": {
          "businessRole": "semantic metric entity",
          "grain": "one recipient Prefecture x GiftCategory pair",
          "identityProperty": "PrefCategoryMetricId",
          "keyType": "String",
          "snapshotScope": "current static Donation only",
          "supplierPolicy": "excluded",
          "defaultAggregation": "none"
        }
      },
      "properties": {
        "PrefCategoryMetricId": {
          "valueType": "String",
          "isKey": true,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Synthetic stable identifier for one recipient Prefecture x GiftCategory snapshot metric row, encoded as {PrefectureId:02}-{CategoryId:02}, for example 13-01. The measure values are separate properties on this entity. The identical NN-NN text can also be a PrefDonationFlowId, so the entity type - not the string shape - disambiguates the key; always qualify it as a PrefectureCategoryMetric identifier.",
            "customAttributes": {
              "businessRole": "composite metric-row identifier",
              "grain": "one recipient Prefecture x GiftCategory pair",
              "format": "string ID",
              "defaultAggregation": "none",
              "keyEncoding": "{PrefectureId:02}-{CategoryId:02}",
              "keyExample": "13-01",
              "keyAmbiguity": "the same NN-NN text may exist in another metric entity type; qualify by entity type"
            }
          }
        },
        "PrefCategoryStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated static Donation count for one recipient Prefecture x GiftCategory pair.",
            "customAttributes": {
              "businessRole": "cross-dimensional snapshot metric",
              "grain": "one recipient Prefecture x GiftCategory pair",
              "measure": "count of Donation",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefCategoryTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated static Donation amount for one recipient Prefecture x GiftCategory pair.",
            "customAttributes": {
              "businessRole": "cross-dimensional snapshot metric",
              "grain": "one recipient Prefecture x GiftCategory pair",
              "unit": "JPY",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefCategoryAmountRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Amount rank of this GiftCategory inside one recipient Prefecture. Deterministic row-number rank (row_number) over PrefCategoryTotalYen desc, CategoryId asc within each PrefectureId: equal amounts never share a rank because CategoryId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "cross-dimensional amount rank",
              "grain": "one recipient Prefecture x GiftCategory pair",
              "rankScope": "within recipient Prefecture",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "PrefCategoryTotalYen desc, CategoryId asc",
              "rankTieBreak": "CategoryId ascending",
              "rankPartition": "PrefectureId"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "PrefectureDonationFlow": {
      "semanticEnrichment": {
        "synonyms": [
          "prefecture donation flow",
          "origin destination donation flow",
          "居住地から寄付先への寄付フロー",
          "都道府県間寄付フロー"
        ],
        "description": "Pre-aggregated current-static-snapshot flow from Donor residence Prefecture to Donation recipient Prefecture. Origin and destination are not interchangeable. It excludes time-series observations and Suppliers.",
        "customAttributes": {
          "businessRole": "semantic flow metric entity",
          "grain": "one residence Prefecture x recipient Prefecture pair",
          "identityProperty": "PrefDonationFlowId",
          "keyType": "String",
          "direction": "origin residence to destination recipient",
          "snapshotScope": "current static Donation only",
          "supplierPolicy": "excluded",
          "defaultAggregation": "none"
        }
      },
      "properties": {
        "PrefDonationFlowId": {
          "valueType": "String",
          "isKey": true,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Synthetic stable identifier for one donor-residence Prefecture x recipient Prefecture snapshot flow row, encoded as {ResidencePrefectureId:02}-{RecipientPrefectureId:02}, for example 13-01, which reads as residence 13 to recipient 01. Directional measures are separate properties. The identical NN-NN text can also be a PrefCategoryMetricId, so the entity type - not the string shape - disambiguates the key; always qualify it as a PrefectureDonationFlow identifier.",
            "customAttributes": {
              "businessRole": "origin-destination metric-row identifier",
              "grain": "one residence Prefecture x recipient Prefecture pair",
              "format": "string ID",
              "defaultAggregation": "none",
              "keyEncoding": "{ResidencePrefectureId:02}-{RecipientPrefectureId:02}",
              "keyExample": "13-01 = residence Prefecture 13 to recipient Prefecture 01",
              "keyAmbiguity": "the same NN-NN text may exist in another metric entity type; qualify by entity type"
            }
          }
        },
        "PrefFlowStaticCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated count of static Donations from one residence Prefecture to one recipient Prefecture.",
            "customAttributes": {
              "businessRole": "origin-destination snapshot metric",
              "grain": "one residence Prefecture x recipient Prefecture pair",
              "measure": "count of Donation",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefFlowTotalYen": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Pre-aggregated JPY amount from one residence Prefecture to one recipient Prefecture.",
            "customAttributes": {
              "businessRole": "origin-destination snapshot metric",
              "grain": "one residence Prefecture x recipient Prefecture pair",
              "unit": "JPY",
              "defaultAggregation": "none"
            }
          }
        },
        "PrefFlowOriginRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Amount rank of this recipient Prefecture among the destinations of one donor-residence Prefecture: the window is partitioned by the origin, so rank 1 is the destination that received the most from that origin. Deterministic row-number rank (row_number) over PrefFlowTotalYen desc, RecipientPrefectureId asc within each ResidencePrefectureId: equal amounts never share a rank because RecipientPrefectureId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "flow amount rank",
              "grain": "one residence-to-recipient pair",
              "rankScope": "destinations within one residence Prefecture",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "PrefFlowTotalYen desc, RecipientPrefectureId asc",
              "rankTieBreak": "RecipientPrefectureId ascending",
              "rankPartition": "ResidencePrefectureId"
            }
          }
        },
        "PrefFlowDestinationRank": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Amount rank of this donor-residence Prefecture among the origins of one recipient Prefecture: the window is partitioned by the destination, so rank 1 is the origin that sent the most to that destination. Deterministic row-number rank (row_number) over PrefFlowTotalYen desc, ResidencePrefectureId asc within each RecipientPrefectureId: equal amounts never share a rank because ResidencePrefectureId breaks the tie ascending.",
            "customAttributes": {
              "businessRole": "flow amount rank",
              "grain": "one residence-to-recipient pair",
              "rankScope": "origins within one recipient Prefecture",
              "defaultAggregation": "none",
              "rankMethod": "row_number",
              "rankSort": "PrefFlowTotalYen desc, ResidencePrefectureId asc",
              "rankTieBreak": "ResidencePrefectureId ascending",
              "rankPartition": "RecipientPrefectureId"
            }
          }
        }
      },
      "timeseriesProperties": {}
    },
    "Supplier": {
      "semanticEnrichment": {
        "synonyms": [
          "supplier",
          "suppliers",
          "vendor",
          "vendors",
          "provider",
          "providers",
          "merchant",
          "local business",
          "return-gift provider",
          "gift supplier",
          "事業者",
          "提供事業者",
          "返礼品事業者",
          "ベンダー",
          "加盟店"
        ],
        "description": "Synthetic workshop business that may provide zero or more return-gift options (返礼品提供事業者). It is neither the Donation recipient nor a Donor; location is defined by SupplierInPrefecture.",
        "customAttributes": {
          "businessRole": "provider dimension",
          "grain": "one synthetic supplier",
          "sensitivity": "Synthetic-Organization",
          "locationRole": "supplier base prefecture",
          "metricPolicy": "supplier allocation is not defined"
        }
      },
      "properties": {
        "SupplierId": {
          "valueType": "BigInt",
          "isKey": true,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Stable numeric identifier for one synthetic registered Supplier entity. It identifies a catalog provider, not an actual fulfillment event.",
            "customAttributes": {
              "businessRole": "stable entity identifier",
              "grain": "one Supplier",
              "format": "integer ID",
              "fulfillmentRole": "none",
              "defaultAggregation": "none"
            }
          }
        },
        "SupplierName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Synthetic Japanese organization-style name of the Supplier or vendor (事業者名／提供事業者名). It is not a Donation recipient.",
            "customAttributes": {
              "businessRole": "Japanese label",
              "grain": "one value per Supplier",
              "language": "ja-JP",
              "format": "synthetic Japanese organization name",
              "sensitivity": "Synthetic-Organization",
              "defaultAggregation": "none"
            }
          }
        },
        "SupplierDisplayName": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": true,
          "semanticEnrichment": {
            "description": "Display-only composite in the format {SupplierName} / Supplier {SupplierId}. Use SupplierId for identity and this property for human-readable output.",
            "customAttributes": {
              "businessRole": "display label",
              "grain": "one value per Supplier",
              "format": "{SupplierName} / Supplier {SupplierId}",
              "language": "ja-JP",
              "sensitivity": "Synthetic-Organization",
              "defaultAggregation": "none"
            }
          }
        },
        "SupplierType": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Japanese controlled supplier classification or business-type label (事業者種別／事業者区分) for the Supplier or vendor. It describes the provider, not the Gift category or Donation recipient.",
            "customAttributes": {
              "businessRole": "Japanese categorical attribute",
              "grain": "one value per Supplier",
              "language": "ja-JP",
              "format": "controlled text, 28 values",
              "sensitivity": "Synthetic-Organization",
              "defaultAggregation": "none"
            }
          }
        },
        "SupplierTypeEn": {
          "valueType": "String",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Canonical English business-type label for SupplierType. Use it for English filtering and grouping; it describes the provider and not the Gift category or Donation recipient.",
            "customAttributes": {
              "businessRole": "canonical English categorical attribute",
              "grain": "one value per Supplier",
              "language": "en",
              "format": "controlled text, 28 values",
              "sensitivity": "Synthetic-Organization",
              "defaultAggregation": "none"
            }
          }
        },
        "SupplierProvidedGiftCount": {
          "valueType": "BigInt",
          "isKey": false,
          "isDisplayName": false,
          "semanticEnrichment": {
            "description": "Count of distinct Gifts linked to this Supplier in the registered catalog; it is not a Donation or fulfillment metric.",
            "customAttributes": {
              "businessRole": "catalog relationship count",
              "grain": "one Supplier",
              "measure": "count of registered Gift links",
              "monetaryRole": "none",
              "fulfillmentRole": "none",
              "defaultAggregation": "none"
            }
          }
        }
      },
      "timeseriesProperties": {}
    }
  },
  "relationships": {
    "DonationSelectedGift": {
      "sourceEntityType": "Donation",
      "targetEntityType": "Gift",
      "semanticEnrichment": {
        "description": "Connects each static baseline Donation to the selected Gift option. A no-return-gift choice is represented by a Gift option in the no-return-gift category, not by a missing Donation.",
        "customAttributes": {
          "direction": "Donation -> Gift",
          "businessRole": "transaction selection",
          "cardinality": "many-to-one, exactly one Gift option per Donation in this dataset",
          "grain": "one edge per Donation",
          "sensitivity": "Synthetic-Transaction",
          "aggregationGuard": "aggregate DonationAmountYen from Donation, not Gift",
          "sourceParticipation": "mandatory - every Donation carries exactly one edge",
          "targetParticipation": "optional - a Gift may have zero Donation rows"
        }
      }
    },
    "DonationToMunicipality": {
      "sourceEntityType": "Donation",
      "targetEntityType": "Municipality",
      "semanticEnrichment": {
        "description": "Connects each static baseline Donation to exactly one recipient Municipality. This is the authoritative path for received-by geography and must not be replaced by Gift or Supplier location.",
        "customAttributes": {
          "direction": "Donation -> Municipality",
          "businessRole": "recipient geography",
          "cardinality": "many-to-one, exactly one Municipality per Donation",
          "grain": "one edge per Donation",
          "sensitivity": "Synthetic-Transaction",
          "aggregationGuard": "authoritative recipient path for Donation counts and amounts",
          "sourceParticipation": "mandatory - every Donation carries exactly one edge",
          "targetParticipation": "optional - a Municipality may have zero Donation rows"
        }
      }
    },
    "DonorLivesInPrefecture": {
      "sourceEntityType": "Donor",
      "targetEntityType": "Prefecture",
      "semanticEnrichment": {
        "description": "Connects a synthetic Donor to the Prefecture of donor residence. This relationship defines donor-origin geography only and must never be treated as the Donation recipient.",
        "customAttributes": {
          "direction": "Donor -> Prefecture",
          "businessRole": "donor residence geography",
          "cardinality": "many-to-one, exactly one Prefecture per Donor",
          "grain": "one edge per Donor",
          "sensitivity": "Synthetic-Personal",
          "aggregationGuard": "use DonorMadeDonation before aggregating Donation facts",
          "sourceParticipation": "mandatory - every Donor carries exactly one edge",
          "targetParticipation": "optional - a Prefecture may have zero Donor rows"
        }
      }
    },
    "DonorMadeDonation": {
      "sourceEntityType": "Donor",
      "targetEntityType": "Donation",
      "semanticEnrichment": {
        "description": "Connects the Donor associated with a static baseline Donation to that Donation. Every static Donation has exactly one Donor, and a Donor may have zero, one or many static Donations: 18 of the 12,000 Donors made none.",
        "customAttributes": {
          "direction": "Donor -> Donation",
          "businessRole": "transaction actor",
          "cardinality": "zero-to-many from Donor, exactly one Donor per Donation",
          "grain": "one edge per Donation",
          "sensitivity": "Synthetic-Transaction",
          "aggregationGuard": "count target Donation entities and sum their DonationAmountYen",
          "sourceParticipation": "optional - 18 Donors have no Donation",
          "targetParticipation": "mandatory - every Donation has exactly one Donor"
        }
      }
    },
    "FlowToRecipientPref": {
      "sourceEntityType": "PrefectureDonationFlow",
      "targetEntityType": "Prefecture",
      "semanticEnrichment": {
        "description": "Connects a PrefectureDonationFlow row to the Donation-recipient Prefecture.",
        "customAttributes": {
          "direction": "PrefectureDonationFlow -> recipient Prefecture",
          "businessRole": "Donation-recipient destination of a pre-aggregated prefecture flow",
          "geographyRole": "destination recipient",
          "cardinality": "many-to-one, exactly one recipient Prefecture per PrefectureDonationFlow row",
          "grain": "one edge per flow row",
          "sensitivity": "Public",
          "aggregationGuard": "flow rows are already aggregated at residence x recipient Prefecture grain",
          "sourceParticipation": "mandatory - every PrefectureDonationFlow carries exactly one edge",
          "targetParticipation": "optional - a Prefecture may have zero PrefectureDonationFlow rows"
        }
      }
    },
    "GiftInCategory": {
      "sourceEntityType": "Gift",
      "targetEntityType": "GiftCategory",
      "semanticEnrichment": {
        "description": "Classifies a Gift in exactly one GiftCategory. Category-level donation metrics must traverse DonationSelectedGift first and aggregate DonationAmountYen from Donation.",
        "customAttributes": {
          "direction": "Gift -> GiftCategory",
          "businessRole": "catalog classification",
          "cardinality": "many-to-one, exactly one GiftCategory per Gift",
          "grain": "one edge per Gift",
          "sensitivity": "Public-Synthetic-Catalog",
          "aggregationGuard": "aggregate Donation facts, not Gift or edge counts",
          "sourceParticipation": "mandatory - every Gift carries exactly one edge",
          "targetParticipation": "optional - a GiftCategory may have zero Gift rows"
        }
      }
    },
    "MunHasCategoryMetric": {
      "sourceEntityType": "Municipality",
      "targetEntityType": "MunicipalityCategoryMetric",
      "semanticEnrichment": {
        "description": "Connects a Municipality to each MunicipalityCategoryMetric row for that recipient Municipality. A Municipality that received no Donation has no metric row at all: 10 of the 1,741 Municipalities have none.",
        "customAttributes": {
          "direction": "Municipality -> MunicipalityCategoryMetric",
          "businessRole": "recipient Municipality to pre-aggregated category metric",
          "cardinality": "zero-to-many from Municipality, exactly one Municipality per MunicipalityCategoryMetric row",
          "grain": "one edge per metric row",
          "sensitivity": "Public",
          "aggregationGuard": "read metric values once and do not traverse to Donation detail for the same total",
          "sourceParticipation": "optional - 10 Municipalities have no metric row",
          "targetParticipation": "mandatory - every metric row has exactly one Municipality"
        }
      }
    },
    "MunMetricForCategory": {
      "sourceEntityType": "MunicipalityCategoryMetric",
      "targetEntityType": "GiftCategory",
      "semanticEnrichment": {
        "description": "Connects a MunicipalityCategoryMetric row to its GiftCategory.",
        "customAttributes": {
          "direction": "MunicipalityCategoryMetric -> GiftCategory",
          "businessRole": "pre-aggregated category metric to its GiftCategory",
          "cardinality": "many-to-one, exactly one GiftCategory per MunicipalityCategoryMetric row",
          "grain": "one edge per metric row",
          "sensitivity": "Public",
          "aggregationGuard": "metric is already aggregated at Municipality x GiftCategory grain",
          "sourceParticipation": "mandatory - every MunicipalityCategoryMetric carries exactly one edge",
          "targetParticipation": "optional - a GiftCategory may have zero MunicipalityCategoryMetric rows"
        }
      }
    },
    "MunicipalityCatalogsGift": {
      "sourceEntityType": "Municipality",
      "targetEntityType": "Gift",
      "semanticEnrichment": {
        "description": "Connects a Municipality to each Gift registered in that Municipality's Workshop catalog. This is catalog geography, not Donation recipient geography, Supplier location, fulfillment location, or shipping origin. In this synthetic dataset every Donation happens to select a Gift catalogued by its own recipient Municipality, so a forbidden count taken through this edge can coincidentally return the same number as the correct DonationToMunicipality count. That coincidence is a property of the sample data, not of the model: the rule stays, because real catalogues diverge and the two paths answer different questions.",
        "customAttributes": {
          "direction": "Municipality -> Gift",
          "businessRole": "gift catalog association",
          "cardinality": "one-to-many from Municipality, exactly one catalog Municipality per Gift",
          "grain": "one edge per Gift",
          "sensitivity": "Public-Synthetic-Catalog",
          "aggregationGuard": "never count or sum Donation facts through this relationship; use DonationToMunicipality for received-by geography",
          "sourceParticipation": "mandatory - every one of the 1,741 Municipalities catalogs at least one Gift",
          "targetParticipation": "mandatory - every Gift has exactly one catalog Municipality",
          "dataCoincidenceWarning": "in this dataset the forbidden catalog path can return the same total as the correct recipient path; the guard protects future divergence, not this sample"
        }
      }
    },
    "MunicipalityInPrefecture": {
      "sourceEntityType": "Municipality",
      "targetEntityType": "Prefecture",
      "semanticEnrichment": {
        "description": "Connects each recipient Municipality to its containing Prefecture. Follow Municipality to Prefecture for recipient geography; do not use this relationship to infer donor residence or to count Donations.",
        "customAttributes": {
          "direction": "Municipality -> Prefecture",
          "businessRole": "recipient administrative containment",
          "cardinality": "many-to-one, exactly one Prefecture per Municipality",
          "grain": "one edge per Municipality",
          "sensitivity": "Public",
          "aggregationGuard": "count and sum Donation entities, not edges",
          "sourceParticipation": "mandatory - every Municipality carries exactly one edge",
          "targetParticipation": "optional - a Prefecture may have zero Municipality rows"
        }
      }
    },
    "PrefHasCategoryMetric": {
      "sourceEntityType": "Prefecture",
      "targetEntityType": "PrefectureCategoryMetric",
      "semanticEnrichment": {
        "description": "Connects a recipient Prefecture to each PrefectureCategoryMetric row for that Prefecture.",
        "customAttributes": {
          "direction": "Prefecture -> PrefectureCategoryMetric",
          "businessRole": "recipient Prefecture to pre-aggregated category metric",
          "geographyRole": "recipient",
          "cardinality": "one-to-many from Prefecture, exactly one recipient Prefecture per PrefectureCategoryMetric row",
          "grain": "one edge per metric row",
          "sensitivity": "Public",
          "aggregationGuard": "read metric values once and do not traverse to Donation detail for the same total",
          "sourceParticipation": "mandatory - every one of the 47 Prefectures has at least one metric row",
          "targetParticipation": "mandatory - every metric row has exactly one recipient Prefecture"
        }
      }
    },
    "PrefMetricForCategory": {
      "sourceEntityType": "PrefectureCategoryMetric",
      "targetEntityType": "GiftCategory",
      "semanticEnrichment": {
        "description": "Connects a PrefectureCategoryMetric row to its GiftCategory.",
        "customAttributes": {
          "direction": "PrefectureCategoryMetric -> GiftCategory",
          "businessRole": "pre-aggregated category metric to its GiftCategory",
          "cardinality": "many-to-one, exactly one GiftCategory per PrefectureCategoryMetric row",
          "grain": "one edge per metric row",
          "sensitivity": "Public",
          "aggregationGuard": "metric is already aggregated at recipient Prefecture x GiftCategory grain",
          "sourceParticipation": "mandatory - every PrefectureCategoryMetric carries exactly one edge",
          "targetParticipation": "optional - a GiftCategory may have zero PrefectureCategoryMetric rows"
        }
      }
    },
    "ResidencePrefHasFlow": {
      "sourceEntityType": "Prefecture",
      "targetEntityType": "PrefectureDonationFlow",
      "semanticEnrichment": {
        "description": "Connects the Donor-residence Prefecture to an origin-destination PrefectureDonationFlow row.",
        "customAttributes": {
          "direction": "residence Prefecture -> PrefectureDonationFlow",
          "businessRole": "donor-residence origin of a pre-aggregated prefecture flow",
          "geographyRole": "origin donor residence",
          "cardinality": "one-to-many from Prefecture, exactly one residence Prefecture per PrefectureDonationFlow row",
          "grain": "one edge per flow row",
          "sensitivity": "Public",
          "aggregationGuard": "flow rows are already aggregated at residence x recipient Prefecture grain",
          "sourceParticipation": "mandatory - every one of the 47 Prefectures originates at least one flow row",
          "targetParticipation": "mandatory - every flow row has exactly one residence Prefecture"
        }
      }
    },
    "SupplierInPrefecture": {
      "sourceEntityType": "Supplier",
      "targetEntityType": "Prefecture",
      "semanticEnrichment": {
        "description": "Connects a synthetic Supplier to its registered Prefecture. Registered Supplier location is neither donor residence, Donation recipient geography, fulfillment location, service location, nor shipping origin.",
        "customAttributes": {
          "direction": "Supplier -> Prefecture",
          "businessRole": "supplier base geography",
          "cardinality": "many-to-one, exactly one Prefecture per Supplier",
          "grain": "one edge per Supplier",
          "sensitivity": "Synthetic-Organization",
          "aggregationGuard": "do not infer Donation recipient from Supplier location",
          "sourceParticipation": "mandatory - every Supplier carries exactly one edge",
          "targetParticipation": "optional - a Prefecture may have zero Supplier rows"
        }
      }
    },
    "SupplierProvidesGift": {
      "sourceEntityType": "Supplier",
      "targetEntityType": "Gift",
      "semanticEnrichment": {
        "description": "Connects a Supplier, provider, vendor, 事業者, 業者, or 会社 to a Gift for which it is a registered provider. It does not prove an actual fulfillment, service, shipment, or physical origin. For a Gift-centered provider question, resolve the GiftId first and traverse this relationship in reverse from Gift to every registered Supplier, then use SupplierInPrefecture for registered location. The registration contract is many-to-many and optional on both sides: a Gift may have zero, one or many Suppliers and a Supplier may register zero, one or many Gifts. In this dataset every one of the 2,500 Suppliers and all 6,000 Gifts participate, across 14,514 unique pairs, so no orphan appears at run time. No allocation of DonationAmountYen among Suppliers is defined.",
        "customAttributes": {
          "direction": "Supplier -> Gift",
          "businessRole": "catalog supply relationship",
          "cardinality": "many-to-many, zero-to-many on both sides by contract",
          "grain": "one unique Supplier-Gift pair",
          "sensitivity": "Public-Synthetic-Catalog",
          "aggregationGuard": "do not sum DonationAmountYen across Suppliers without an explicit allocation rule because multi-supplier Gifts can duplicate attribution",
          "sourceParticipation": "optional by contract; total in this dataset - 0 Suppliers without a Gift",
          "targetParticipation": "optional by contract; total in this dataset - 0 Gifts without a Supplier"
        }
      }
    }
  }
}''')

## Validated update engine

The following cell is generated from the repository-tested implementation.
It validates the exact schema, previews hashes, handles Fabric LROs on the
Fabric host, rejects concurrent definition drift, and verifies the final
definition after an approved write.

In [ ]:
from __future__ import annotations

import base64
import copy
import hashlib
import json
import re
import time
from dataclasses import dataclass
from typing import Any, Callable, Sequence
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse


FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
MAX_PAGE_REQUESTS = 1000
ENTITY_PART_PATTERN = re.compile(r"^EntityTypes/([^/]+)/definition\.json$")
RELATIONSHIP_PART_PATTERN = re.compile(
    r"^RelationshipTypes/([^/]+)/definition\.json$"
)
GUID_PATTERN = re.compile(
    r"(?<![0-9a-fA-F])[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-"
    r"[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}(?![0-9a-fA-F])"
)
JWT_PATTERN = re.compile(r"eyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+")
EMAIL_PATTERN = re.compile(
    r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}",
    re.IGNORECASE,
)
PROHIBITED_MANIFEST_KEYS = {
    "id",
    "workspaceId",
    "folderId",
    "itemId",
    "entityTypeId",
    "propertyId",
    "relationshipTypeId",
}
CONFIRMATION_PHRASE = "APPLY ONTOLOGY METADATA"


class OntologyMetadataError(ValueError):
    pass


class FabricApiError(RuntimeError):
    pass


@dataclass(frozen=True)
class MetadataChange:
    scope: str
    name: str
    entity_name: str | None
    previous_hash: str
    desired_hash: str

    @property
    def label(self) -> str:
        if self.entity_name:
            return f"{self.entity_name}.{self.name}"
        return self.name


@dataclass
class MetadataPlan:
    definition: dict[str, Any]
    changes: list[MetadataChange]
    counts: dict[str, int]
    original_digest: str
    patched_digest: str
    structural_digest: str
    manifest_digest: str
    manifest_counts: dict[str, int] | None = None
    entity_names: tuple[str, ...] = ()
    relationship_names: tuple[str, ...] = ()


def _format_issue_report(headline: str, issues: Sequence[str]) -> str:
    """Render every collected issue as one actionable, numbered message."""
    lines = [headline]
    lines.extend(f"  {position}. {issue}" for position, issue in enumerate(issues, start=1))
    return "\n".join(lines)


def _format_alignment_report(issues: Sequence[str]) -> str:
    """Render every collected preflight mismatch as one actionable message."""
    return _format_issue_report(
        f"The Ontology does not match the canonical metadata manifest "
        f"({len(issues)} mismatch(es)). Fix all of them before applying:",
        issues,
    )


def _describe_name_difference(label: str, actual: set[str], desired: set[str]) -> list[str]:
    issues: list[str] = []
    missing = sorted(desired - actual)
    unexpected = sorted(actual - desired)
    if missing:
        issues.append(f"{label} missing from the Ontology: {missing}")
    if unexpected:
        issues.append(f"{label} present in the Ontology but not in the manifest: {unexpected}")
    return issues


@dataclass
class _PartObject:
    part_index: int
    path: str
    value: dict[str, Any]


@dataclass
class _DefinitionIndex:
    entities: dict[str, _PartObject]
    relationships: dict[str, _PartObject]
    entity_id_to_name: dict[str, str]
    static_properties: int
    timeseries_properties: int


def canonical_json_bytes(value: Any) -> bytes:
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")


def sha256_json(value: Any) -> str:
    return hashlib.sha256(canonical_json_bytes(value)).hexdigest()


def normalize_definition(value: dict[str, Any]) -> dict[str, Any]:
    definition = value.get("definition", value)
    if not isinstance(definition, dict):
        raise OntologyMetadataError("Fabric definition response must be an object.")
    parts = definition.get("parts")
    if not isinstance(parts, list) or not parts:
        raise OntologyMetadataError("Fabric definition must contain nonempty parts.")
    return copy.deepcopy(definition)


def _decode_part_json(part: dict[str, Any]) -> dict[str, Any]:
    if part.get("payloadType") != "InlineBase64":
        raise OntologyMetadataError(
            f"Unsupported payloadType for {part.get('path')!r}."
        )
    payload = part.get("payload")
    if not isinstance(payload, str) or not payload:
        raise OntologyMetadataError(f"Missing payload for {part.get('path')!r}.")
    try:
        raw = base64.b64decode(payload, validate=True)
        value = json.loads(raw.decode("utf-8-sig"))
    except (ValueError, UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise OntologyMetadataError(
            f"Invalid JSON payload for {part.get('path')!r}: {exc}"
        ) from exc
    if not isinstance(value, dict):
        raise OntologyMetadataError(
            f"Definition part {part.get('path')!r} must contain an object."
        )
    return value


def ontology_json_bytes(value: Any) -> bytes:
    """Keep the Fabric polymorphic discriminator first, independently of hash order."""
    def ordered(node):
        if isinstance(node, dict):
            keys = (["sourceType"] if "sourceType" in node else []) + [
                key for key in node if key != "sourceType"
            ]
            return {key: ordered(node[key]) for key in keys}
        if isinstance(node, list):
            return [ordered(child) for child in node]
        return node

    return json.dumps(
        ordered(value), ensure_ascii=False, separators=(",", ":"), sort_keys=False,
    ).encode("utf-8")


def _encode_part_json(value: dict[str, Any]) -> str:
    return base64.b64encode(ontology_json_bytes(value)).decode("ascii")


def _index_definition(definition: dict[str, Any]) -> _DefinitionIndex:
    entities: dict[str, _PartObject] = {}
    relationships: dict[str, _PartObject] = {}
    entity_id_to_name: dict[str, str] = {}
    seen_paths: set[str] = set()
    static_properties = 0
    timeseries_properties = 0

    for index, part in enumerate(definition["parts"]):
        if not isinstance(part, dict):
            raise OntologyMetadataError("Every definition part must be an object.")
        path = part.get("path")
        if not isinstance(path, str) or not path:
            raise OntologyMetadataError("Every definition part needs a path.")
        if "\\" in path or path.startswith("/"):
            raise OntologyMetadataError(f"Invalid definition part path: {path!r}.")
        if path in seen_paths:
            raise OntologyMetadataError(f"Duplicate definition part path: {path!r}.")
        seen_paths.add(path)

        entity_match = ENTITY_PART_PATTERN.fullmatch(path)
        if entity_match:
            value = _decode_part_json(part)
            name = value.get("name")
            entity_id = str(value.get("id", ""))
            if not isinstance(name, str) or not name or not entity_id:
                raise OntologyMetadataError(
                    f"Entity definition {path!r} lacks a name or id."
                )
            if entity_match.group(1) != entity_id:
                raise OntologyMetadataError(
                    f"Entity path/id mismatch for {name!r}."
                )
            if name in entities or entity_id in entity_id_to_name:
                raise OntologyMetadataError(
                    f"Duplicate Entity type name or id: {name!r}."
                )
            properties = value.get("properties", [])
            timeseries = value.get("timeseriesProperties", [])
            if not isinstance(properties, list) or not isinstance(timeseries, list):
                raise OntologyMetadataError(
                    f"Entity {name!r} has invalid property collections."
                )
            static_properties += len(properties)
            timeseries_properties += len(timeseries)
            entities[name] = _PartObject(index, path, value)
            entity_id_to_name[entity_id] = name
            continue

        relationship_match = RELATIONSHIP_PART_PATTERN.fullmatch(path)
        if relationship_match:
            value = _decode_part_json(part)
            name = value.get("name")
            relationship_id = str(value.get("id", ""))
            if not isinstance(name, str) or not name or not relationship_id:
                raise OntologyMetadataError(
                    f"Relationship definition {path!r} lacks a name or id."
                )
            if relationship_match.group(1) != relationship_id:
                raise OntologyMetadataError(
                    f"Relationship path/id mismatch for {name!r}."
                )
            if name in relationships:
                raise OntologyMetadataError(
                    f"Duplicate Relationship type name: {name!r}."
                )
            relationships[name] = _PartObject(index, path, value)

    if not entities:
        raise OntologyMetadataError("No Entity type definitions were found.")
    if not relationships:
        raise OntologyMetadataError("No Relationship type definitions were found.")
    return _DefinitionIndex(
        entities=entities,
        relationships=relationships,
        entity_id_to_name=entity_id_to_name,
        static_properties=static_properties,
        timeseries_properties=timeseries_properties,
    )


def _walk_manifest(value: Any) -> None:
    if isinstance(value, dict):
        for key, child in value.items():
            if key in PROHIBITED_MANIFEST_KEYS:
                raise OntologyMetadataError(
                    f"Tenant or definition-specific key is prohibited: {key!r}."
                )
            _walk_manifest(child)
    elif isinstance(value, list):
        for child in value:
            _walk_manifest(child)


def _require_semantic_enrichment(
    value: Any,
    label: str,
) -> dict[str, Any]:
    if not isinstance(value, dict):
        raise OntologyMetadataError(
            f"{label} semanticEnrichment must be an object."
        )
    description = value.get("description")
    if not isinstance(description, str) or not description.strip():
        raise OntologyMetadataError(
            f"{label} semanticEnrichment needs a nonempty description."
        )
    custom_attributes = value.get("customAttributes")
    if custom_attributes is not None and not isinstance(custom_attributes, dict):
        raise OntologyMetadataError(
            f"{label} customAttributes must be an object."
        )
    synonyms = value.get("synonyms")
    if synonyms is not None and (
        not isinstance(synonyms, list)
        or not all(isinstance(item, str) and item.strip() for item in synonyms)
    ):
        raise OntologyMetadataError(f"{label} synonyms must be nonempty strings.")
    return value


# Every relationship must state which way it is traversed, how many rows join on
# each side, and the row grain the edge count follows. A missing cardinality lets
# an agent invent a fan-out, so the manifest is rejected instead of applied.
#
# The accepted vocabulary covers all four shapes the model uses. `many-to-many`
# is one of them: SupplierProvidesGift is a registration bridge, and leaving it
# out of this tuple rejected the whole manifest, so no metadata could be applied
# at all.
CARDINALITY_PREFIXES = (
    "one-to-one",
    "one-to-many from ",
    "zero-to-many from ",
    "many-to-one",
    "many-to-many",
)
#: Multiplicity alone does not say whether a side may contribute zero rows, and
#: the aggregation guards depend on that: a source with no edge is a row an agent
#: must not silently drop. Both participation statements are therefore required
#: alongside the direction, the cardinality and the grain.
REQUIRED_RELATIONSHIP_ATTRIBUTES = (
    "direction",
    "cardinality",
    "grain",
    "sourceParticipation",
    "targetParticipation",
)


def _relationship_attribute_issues(
    name: str,
    enrichment: dict[str, Any],
) -> list[str]:
    attributes = enrichment.get("customAttributes")
    if not isinstance(attributes, dict):
        return [f"Relationship {name}: customAttributes object is missing."]
    issues: list[str] = []
    for key in REQUIRED_RELATIONSHIP_ATTRIBUTES:
        value = attributes.get(key)
        if not isinstance(value, str) or not value.strip():
            issues.append(f"Relationship {name}: customAttributes.{key} is missing.")
    cardinality = attributes.get("cardinality")
    if isinstance(cardinality, str) and cardinality.strip():
        if not cardinality.startswith(CARDINALITY_PREFIXES):
            issues.append(
                f"Relationship {name}: cardinality {cardinality!r} does not start with "
                f"one of {CARDINALITY_PREFIXES}."
            )
        elif cardinality.startswith("one-to-many from "):
            declared_source = cardinality[len("one-to-many from "):].split(",", 1)[0]
            direction = str(attributes.get("direction", ""))
            if declared_source and declared_source not in direction:
                issues.append(
                    f"Relationship {name}: cardinality names {declared_source!r} as the "
                    f"one side but direction is {direction!r}."
                )
    return issues


def validate_manifest(manifest: dict[str, Any]) -> dict[str, int]:
    if not isinstance(manifest, dict):
        raise OntologyMetadataError("Metadata manifest root must be an object.")
    if manifest.get("schemaVersion") != "1.0":
        raise OntologyMetadataError("Unsupported metadata manifest schemaVersion.")
    if manifest.get("packageVersion") != "2.7.0":
        raise OntologyMetadataError("Metadata manifest packageVersion must be 2.7.0.")
    if manifest.get("ontologyDisplayNameTemplate") != "ONT_Furusato_<PID>":
        raise OntologyMetadataError(
            "Metadata manifest must use ONT_Furusato_<PID>."
        )

    _walk_manifest(manifest)
    serialized = json.dumps(manifest, ensure_ascii=False, sort_keys=True)
    if GUID_PATTERN.search(serialized):
        raise OntologyMetadataError("Metadata manifest must not contain GUIDs.")
    if EMAIL_PATTERN.search(serialized) or JWT_PATTERN.search(serialized):
        raise OntologyMetadataError(
            "Metadata manifest must not contain identities or credentials."
        )

    expected = manifest.get("expectedContract")
    entities = manifest.get("entities")
    relationships = manifest.get("relationships")
    if not isinstance(expected, dict):
        raise OntologyMetadataError("expectedContract must be an object.")
    if not isinstance(entities, dict) or not isinstance(relationships, dict):
        raise OntologyMetadataError(
            "entities and relationships must be objects."
        )

    static_count = 0
    timeseries_count = 0
    for entity_name, entity in entities.items():
        if not isinstance(entity_name, str) or not entity_name:
            raise OntologyMetadataError("Entity names must be nonempty strings.")
        if not isinstance(entity, dict):
            raise OntologyMetadataError(
                f"Entity manifest entry {entity_name!r} must be an object."
            )
        _require_semantic_enrichment(
            entity.get("semanticEnrichment"),
            f"Entity {entity_name}",
        )
        for collection_name in ("properties", "timeseriesProperties"):
            properties = entity.get(collection_name)
            if not isinstance(properties, dict):
                raise OntologyMetadataError(
                    f"{entity_name}.{collection_name} must be an object."
                )
            for property_name, prop in properties.items():
                if not isinstance(prop, dict):
                    raise OntologyMetadataError(
                        f"{entity_name}.{property_name} must be an object."
                    )
                if prop.get("valueType") not in {
                    "String",
                    "Boolean",
                    "DateTime",
                    "Object",
                    "BigInt",
                    "Double",
                }:
                    raise OntologyMetadataError(
                        f"Unsupported valueType for {entity_name}.{property_name}."
                    )
                _require_semantic_enrichment(
                    prop.get("semanticEnrichment"),
                    f"Property {entity_name}.{property_name}",
                )
            if collection_name == "properties":
                static_count += len(properties)
            else:
                timeseries_count += len(properties)

    relationship_issues: list[str] = []
    for relationship_name, relationship in relationships.items():
        if not isinstance(relationship, dict):
            raise OntologyMetadataError(
                f"Relationship {relationship_name!r} must be an object."
            )
        source = relationship.get("sourceEntityType")
        target = relationship.get("targetEntityType")
        if source not in entities or target not in entities or source == target:
            raise OntologyMetadataError(
                f"Relationship {relationship_name!r} has invalid endpoints."
            )
        enrichment = _require_semantic_enrichment(
            relationship.get("semanticEnrichment"),
            f"Relationship {relationship_name}",
        )
        relationship_issues.extend(
            _relationship_attribute_issues(relationship_name, enrichment)
        )
    if relationship_issues:
        raise OntologyMetadataError(
            _format_issue_report(
                f"The metadata manifest does not declare complete relationship "
                f"traversal semantics ({len(relationship_issues)} issue(s)). Every "
                f"relationship needs direction, cardinality and grain:",
                relationship_issues,
            )
        )

    counts = {
        "entityTypes": len(entities),
        "staticProperties": static_count,
        "timeseriesProperties": timeseries_count,
        "relationshipTypes": len(relationships),
    }
    if counts != expected:
        raise OntologyMetadataError(
            f"Manifest counts {counts} do not match expectedContract {expected}."
        )
    return counts


def definition_digest(definition: dict[str, Any]) -> str:
    normalized = normalize_definition(definition)
    value = {
        "format": normalized.get("format"),
        "parts": sorted(
            [
                {
                    "path": part.get("path"),
                    "payloadType": part.get("payloadType"),
                    "payload": part.get("payload"),
                }
                for part in normalized["parts"]
            ],
            key=lambda part: str(part["path"]),
        ),
    }
    return sha256_json(value)


def _without_semantic_enrichment(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            key: _without_semantic_enrichment(child)
            for key, child in value.items()
            if key != "semanticEnrichment"
        }
    if isinstance(value, list):
        return [_without_semantic_enrichment(child) for child in value]
    return value


def structural_digest(definition: dict[str, Any]) -> str:
    normalized = normalize_definition(definition)
    parts: list[dict[str, Any]] = []
    for part in normalized["parts"]:
        path = str(part.get("path", ""))
        if ENTITY_PART_PATTERN.fullmatch(path) or RELATIONSHIP_PART_PATTERN.fullmatch(
            path
        ):
            payload_value: Any = _without_semantic_enrichment(
                _decode_part_json(part)
            )
        else:
            payload_value = {
                "payload": part.get("payload"),
                "payloadType": part.get("payloadType"),
            }
        parts.append({"path": path, "value": payload_value})
    return sha256_json(
        {
            "format": normalized.get("format"),
            "parts": sorted(parts, key=lambda part: part["path"]),
        }
    )


def _replace_metadata(
    target: dict[str, Any],
    desired: dict[str, Any],
    *,
    scope: str,
    name: str,
    entity_name: str | None,
    changes: list[MetadataChange],
) -> None:
    current = target.get("semanticEnrichment")
    if current == desired:
        return
    changes.append(
        MetadataChange(
            scope=scope,
            name=name,
            entity_name=entity_name,
            previous_hash=sha256_json(current),
            desired_hash=sha256_json(desired),
        )
    )
    target["semanticEnrichment"] = copy.deepcopy(desired)


def _property_map(
    value: dict[str, Any],
    collection_name: str,
    entity_name: str,
) -> dict[str, dict[str, Any]]:
    properties = value.get(collection_name, [])
    result: dict[str, dict[str, Any]] = {}
    for prop in properties:
        name = prop.get("name")
        if not isinstance(name, str) or not name:
            raise OntologyMetadataError(
                f"{entity_name}.{collection_name} contains an unnamed property."
            )
        if name in result:
            raise OntologyMetadataError(
                f"Duplicate property {entity_name}.{name}."
            )
        result[name] = prop
    return result


def plan_metadata_update(
    definition: dict[str, Any],
    manifest: dict[str, Any],
) -> MetadataPlan:
    manifest_counts = validate_manifest(manifest)
    original = normalize_definition(definition)
    working = copy.deepcopy(original)
    before_structure = structural_digest(original)
    index = _index_definition(working)

    manifest_entities = manifest["entities"]
    manifest_relationships = manifest["relationships"]
    # Collect every structural difference so one preview run reports the whole
    # contract gap. The apply path stays fail-closed: any collected issue raises
    # before a patched definition is produced.
    issues: list[str] = []
    issues.extend(
        _describe_name_difference(
            "Entity types", set(index.entities), set(manifest_entities)
        )
    )
    issues.extend(
        _describe_name_difference(
            "Relationship types", set(index.relationships), set(manifest_relationships)
        )
    )

    actual_counts = {
        "entityTypes": len(index.entities),
        "staticProperties": index.static_properties,
        "timeseriesProperties": index.timeseries_properties,
        "relationshipTypes": len(index.relationships),
    }
    for count_name, expected_count in sorted(manifest_counts.items()):
        actual_count = actual_counts[count_name]
        if actual_count != expected_count:
            issues.append(
                f"Count mismatch for {count_name}: Ontology has {actual_count}, "
                f"manifest expects {expected_count}"
            )

    changes: list[MetadataChange] = []
    for entity_name in sorted(set(manifest_entities) & set(index.entities)):
        part_object = index.entities[entity_name]
        entity = part_object.value
        desired_entity = manifest_entities[entity_name]
        change_count_before_entity = len(changes)
        _replace_metadata(
            entity,
            desired_entity["semanticEnrichment"],
            scope="EntityType",
            name=entity_name,
            entity_name=None,
            changes=changes,
        )

        key_ids = {str(value) for value in entity.get("entityIdParts", [])}
        display_id = str(entity.get("displayNamePropertyId", ""))
        for collection_name, scope in (
            ("properties", "Property"),
            ("timeseriesProperties", "TimeSeriesProperty"),
        ):
            actual_properties = _property_map(
                entity,
                collection_name,
                entity_name,
            )
            desired_properties = desired_entity[collection_name]
            issues.extend(
                _describe_name_difference(
                    f"{entity_name}.{collection_name}",
                    set(actual_properties),
                    set(desired_properties),
                )
            )
            for property_name in sorted(set(desired_properties) & set(actual_properties)):
                actual = actual_properties[property_name]
                desired = desired_properties[property_name]
                if actual.get("valueType") != desired.get("valueType"):
                    issues.append(
                        f"Value type mismatch for {entity_name}.{property_name}: "
                        f"Ontology has {actual.get('valueType')!r}, "
                        f"manifest expects {desired.get('valueType')!r}"
                    )
                if collection_name == "properties":
                    actual_is_key = str(actual.get("id", "")) in key_ids
                    actual_is_display = (
                        str(actual.get("id", "")) == display_id
                    )
                    if actual_is_key != bool(desired.get("isKey")):
                        issues.append(
                            f"Key flag mismatch for {entity_name}.{property_name}: "
                            f"Ontology isKey={actual_is_key}, "
                            f"manifest expects isKey={bool(desired.get('isKey'))}"
                        )
                    if actual_is_display != bool(
                        desired.get("isDisplayName")
                    ):
                        issues.append(
                            f"Display flag mismatch for {entity_name}.{property_name}: "
                            f"Ontology isDisplayName={actual_is_display}, manifest "
                            f"expects isDisplayName={bool(desired.get('isDisplayName'))}"
                        )
                _replace_metadata(
                    actual,
                    desired["semanticEnrichment"],
                    scope=scope,
                    name=property_name,
                    entity_name=entity_name,
                    changes=changes,
                )
        if len(changes) != change_count_before_entity:
            working["parts"][part_object.part_index]["payload"] = (
                _encode_part_json(entity)
            )

    for relationship_name in sorted(set(manifest_relationships) & set(index.relationships)):
        part_object = index.relationships[relationship_name]
        relationship = part_object.value
        desired = manifest_relationships[relationship_name]
        change_count_before_relationship = len(changes)
        source_id = str(relationship.get("source", {}).get("entityTypeId", ""))
        target_id = str(relationship.get("target", {}).get("entityTypeId", ""))
        source_name = index.entity_id_to_name.get(source_id)
        target_name = index.entity_id_to_name.get(target_id)
        if (
            source_name != desired.get("sourceEntityType")
            or target_name != desired.get("targetEntityType")
        ):
            issues.append(
                f"Direction mismatch for Relationship {relationship_name}: Ontology has "
                f"{source_name} -> {target_name}, manifest expects "
                f"{desired.get('sourceEntityType')} -> {desired.get('targetEntityType')}"
            )
        _replace_metadata(
            relationship,
            desired["semanticEnrichment"],
            scope="RelationshipType",
            name=relationship_name,
            entity_name=None,
            changes=changes,
        )
        if len(changes) != change_count_before_relationship:
            working["parts"][part_object.part_index]["payload"] = (
                _encode_part_json(relationship)
            )

    if issues:
        raise OntologyMetadataError(_format_alignment_report(issues))

    after_structure = structural_digest(working)
    if before_structure != after_structure:
        raise OntologyMetadataError(
            "Metadata planning changed the Ontology structure."
        )
    return MetadataPlan(
        definition=working,
        changes=changes,
        counts=actual_counts,
        original_digest=definition_digest(original),
        patched_digest=definition_digest(working),
        structural_digest=before_structure,
        manifest_digest=sha256_json(manifest),
        manifest_counts=dict(manifest_counts),
        entity_names=tuple(sorted(manifest_entities)),
        relationship_names=tuple(sorted(manifest_relationships)),
    )


def verify_applied_metadata(
    definition: dict[str, Any],
    manifest: dict[str, Any],
    expected_structural_digest: str | None = None,
) -> dict[str, Any]:
    plan = plan_metadata_update(definition, manifest)
    if plan.changes:
        labels = ", ".join(
            f"{change.scope} {change.label}" for change in plan.changes
        )
        raise OntologyMetadataError(
            f"Metadata verification found {len(plan.changes)} remaining changes: "
            f"{labels}"
        )
    if (
        expected_structural_digest is not None
        and plan.structural_digest != expected_structural_digest
    ):
        raise OntologyMetadataError(
            "Ontology structure changed while applying metadata."
        )
    return {
        "verified": True,
        "counts": plan.counts,
        "manifestDigest": plan.manifest_digest,
        "definitionDigest": plan.original_digest,
        "structuralDigest": plan.structural_digest,
    }


def render_change_preview(plan: MetadataPlan) -> list[str]:
    counts: dict[str, int] = {}
    for change in plan.changes:
        counts[change.scope] = counts.get(change.scope, 0) + 1
    expected = plan.manifest_counts or plan.counts
    lines = [
        "Ontology semantic metadata preview",
        "  Preflight contract (Ontology == manifest for every line below):",
        f"    Entity types: {plan.counts['entityTypes']} (manifest {expected['entityTypes']})",
        f"    Static properties: {plan.counts['staticProperties']} "
        f"(manifest {expected['staticProperties']})",
        f"    Time-series properties: {plan.counts['timeseriesProperties']} "
        f"(manifest {expected['timeseriesProperties']})",
        f"    Relationship types: {plan.counts['relationshipTypes']} "
        f"(manifest {expected['relationshipTypes']})",
        "    Entity and Relationship names, property names, value types, key and "
        "display flags, and Relationship directions are all verified before this "
        "preview prints.",
    ]
    if plan.entity_names:
        lines.append(f"    Verified Entity types: {', '.join(plan.entity_names)}")
    if plan.relationship_names:
        lines.append(
            f"    Verified Relationship types: {', '.join(plan.relationship_names)}"
        )
    lines.append(f"  Planned metadata changes: {len(plan.changes)}")
    if counts:
        lines.append(
            "  Change scopes: "
            + ", ".join(f"{key}={counts[key]}" for key in sorted(counts))
        )
    for change in plan.changes:
        lines.append(
            f"  - {change.scope} {change.label}: "
            f"{change.previous_hash[:12]} -> {change.desired_hash[:12]}"
        )
    if not plan.changes:
        lines.append("  - No changes; metadata is already current.")
    lines.extend(
        [
            f"  Manifest SHA-256: {plan.manifest_digest}",
            f"  Structure SHA-256: {plan.structural_digest}",
        ]
    )
    return lines


def redact_error_text(value: str) -> str:
    value = JWT_PATTERN.sub("<redacted-jwt>", value)
    value = GUID_PATTERN.sub("<redacted-guid>", value)
    value = EMAIL_PATTERN.sub("<redacted-email>", value)
    value = re.sub(
        r"(?i)([?&](?:sig|token|access_token|se|sp|sv)=)[^&\s\"'<>]+",
        r"\1<redacted>",
        value,
    )
    return value


class FabricApiClient:
    def __init__(
        self,
        token_provider: Callable[[], str],
        *,
        session: Any | None = None,
        sleep: Callable[[float], None] = time.sleep,
        timeout_seconds: int = 120,
        operation_timeout_seconds: int = 600,
        base_url: str = FABRIC_API_BASE,
    ) -> None:
        if session is None:
            import requests

            session = requests.Session()
        self._token_provider = token_provider
        self._session = session
        self._sleep = sleep
        self._timeout_seconds = timeout_seconds
        self._operation_timeout_seconds = operation_timeout_seconds
        self._base_url = base_url.rstrip("/")
        self._token: str | None = None

    def _get_token(self, refresh: bool = False) -> str:
        if refresh or not self._token:
            token = self._token_provider()
            if not isinstance(token, str) or not token.strip():
                raise FabricApiError("Token provider returned an empty token.")
            self._token = token.strip()
        return self._token

    def _headers(self, refresh: bool = False) -> dict[str, str]:
        return {
            "Authorization": f"Bearer {self._get_token(refresh)}",
            "Accept": "application/json",
            "Content-Type": "application/json",
        }

    @staticmethod
    def _header(response: Any, name: str) -> str:
        headers = getattr(response, "headers", {}) or {}
        for key, value in headers.items():
            if str(key).lower() == name.lower():
                return str(value)
        return ""

    @staticmethod
    def _json(response: Any) -> dict[str, Any]:
        text = getattr(response, "text", "") or ""
        if not text.strip():
            return {}
        try:
            value = response.json()
        except Exception as exc:
            raise FabricApiError("Fabric returned non-JSON content.") from exc
        if not isinstance(value, dict):
            raise FabricApiError("Fabric JSON response must be an object.")
        return value

    def _request(
        self,
        method: str,
        url: str,
        *,
        expected: set[int],
        json_body: dict[str, Any] | None = None,
        idempotent: bool,
    ) -> Any:
        refreshed = False
        attempt = 0
        while True:
            attempt += 1
            try:
                response = self._session.request(
                    method,
                    url,
                    headers=self._headers(refresh=refreshed),
                    json=json_body,
                    timeout=self._timeout_seconds,
                )
            except Exception as exc:
                if idempotent and attempt <= 4:
                    self._sleep(min(2**attempt, 15))
                    continue
                raise FabricApiError(
                    f"Fabric request failed before a response: {type(exc).__name__}."
                ) from exc

            status = int(response.status_code)
            if idempotent and status == 401 and not refreshed:
                refreshed = True
                continue
            retryable = idempotent and status in {429, 500, 502, 503, 504}
            if retryable and attempt <= 4:
                retry_after = self._header(response, "Retry-After")
                delay = int(retry_after) if retry_after.isdigit() else min(
                    2**attempt,
                    15,
                )
                self._sleep(max(delay, 1))
                continue
            if status not in expected:
                body = redact_error_text((getattr(response, "text", "") or "")[:2000])
                raise FabricApiError(f"Fabric HTTP {status}: {body}")
            return response

    def _safe_url(self, value: str) -> str:
        if value.startswith("/"):
            return f"https://api.fabric.microsoft.com{value}"
        parsed = urlparse(value)
        if (
            parsed.scheme != "https"
            or parsed.netloc.lower() != "api.fabric.microsoft.com"
        ):
            raise FabricApiError("Fabric continuation URL used an unexpected host.")
        return value

    def _paged(self, url: str) -> list[dict[str, Any]]:
        values: list[dict[str, Any]] = []
        initial_url = url
        visited: set[str] = set()
        while url:
            if url in visited:
                raise FabricApiError(
                    "Fabric paging repeated a continuation page and was stopped."
                )
            visited.add(url)
            if len(visited) > MAX_PAGE_REQUESTS:
                raise FabricApiError(
                    f"Fabric paging exceeded {MAX_PAGE_REQUESTS} pages."
                )
            response = self._request(
                "GET",
                url,
                expected={200},
                idempotent=True,
            )
            body = self._json(response)
            page = body.get("value", [])
            if not isinstance(page, list) or not all(
                isinstance(item, dict) for item in page
            ):
                raise FabricApiError("Fabric paged response has an invalid value.")
            values.extend(page)
            continuation_uri = body.get("continuationUri")
            if continuation_uri:
                url = self._safe_url(str(continuation_uri))
                continue
            continuation_token = body.get("continuationToken")
            if continuation_token:
                parsed = urlparse(initial_url)
                query = dict(parse_qsl(parsed.query, keep_blank_values=True))
                query["continuationToken"] = str(continuation_token)
                url = urlunparse(
                    parsed._replace(query=urlencode(query))
                )
            else:
                url = ""
        return values

    def resolve_ontology(
        self,
        workspace_id: str,
        display_name: str,
    ) -> dict[str, Any]:
        items = self._paged(f"{self._base_url}/workspaces/{workspace_id}/items")
        matches = [
            item
            for item in items
            if item.get("type") == "Ontology"
            and item.get("displayName") == display_name
        ]
        if len(matches) != 1:
            raise FabricApiError(
                f"Expected one exact Ontology named {display_name!r}; "
                f"found {len(matches)}."
            )
        return matches[0]

    def _wait_operation(
        self,
        initial_response: Any,
        *,
        expect_result: bool,
    ) -> dict[str, Any]:
        operation_id = self._header(initial_response, "x-ms-operation-id")
        if not operation_id:
            raise FabricApiError(
                "Fabric returned 202 without x-ms-operation-id."
            )
        deadline = time.monotonic() + self._operation_timeout_seconds
        retry_after = self._header(initial_response, "Retry-After")
        delay = int(retry_after) if retry_after.isdigit() else 2
        while time.monotonic() < deadline:
            self._sleep(max(delay, 1))
            response = self._request(
                "GET",
                f"{self._base_url}/operations/{operation_id}",
                expected={200, 202},
                idempotent=True,
            )
            poll_retry_after = self._header(response, "Retry-After")
            if poll_retry_after.isdigit():
                delay = int(poll_retry_after)
            if int(response.status_code) == 202:
                continue
            body = self._json(response)
            status = body.get("status")
            if status in {"Failed", "Cancelled", "Canceled"}:
                error = redact_error_text(
                    json.dumps(body.get("error", {}), ensure_ascii=False)[:2000]
                )
                raise FabricApiError(
                    f"Fabric operation ended with {status}: {error}"
                )
            if status not in {"Succeeded", "Completed"}:
                if not status:
                    raise FabricApiError(
                        "Fabric operation returned no terminal status."
                    )
                continue
            if not expect_result:
                return body
            result_response = self._request(
                "GET",
                f"{self._base_url}/operations/{operation_id}/result",
                expected={200},
                idempotent=True,
            )
            return self._json(result_response)
        raise FabricApiError("Fabric operation timed out.")

    def get_definition(
        self,
        workspace_id: str,
        item_id: str,
    ) -> dict[str, Any]:
        response = self._request(
            "POST",
            f"{self._base_url}/workspaces/{workspace_id}/ontologies/"
            f"{item_id}/getDefinition",
            expected={200, 202},
            idempotent=True,
        )
        if int(response.status_code) == 202:
            result = self._wait_operation(response, expect_result=True)
        else:
            result = self._json(response)
        return normalize_definition(result)

    def update_definition(
        self,
        workspace_id: str,
        item_id: str,
        definition: dict[str, Any],
    ) -> dict[str, Any]:
        self._get_token(refresh=True)
        response = self._request(
            "POST",
            f"{self._base_url}/workspaces/{workspace_id}/ontologies/"
            f"{item_id}/updateDefinition",
            expected={200, 202},
            json_body={"definition": definition},
            idempotent=False,
        )
        if int(response.status_code) == 202:
            return self._wait_operation(response, expect_result=False)
        return self._json(response)

## Resolve the current workspace and Ontology, then preview

Run with `APPLY_CHANGES = False` first. Review every planned object and the
manifest / structure hashes. The Notebook prints names only and never prints
tokens or Fabric IDs.

In [ ]:
context = notebookutils.runtime.context
workspace_id = context["currentWorkspaceId"]
workspace_name = context["currentWorkspaceName"]

if EXPECTED_WORKSPACE_NAME and workspace_name != EXPECTED_WORKSPACE_NAME:
    raise OntologyMetadataError(
        f"Workspace mismatch: expected {EXPECTED_WORKSPACE_NAME!r}, "
        f"got {workspace_name!r}."
    )

client = FabricApiClient(
    lambda: notebookutils.credentials.getToken("pbi"),
    operation_timeout_seconds=OPERATION_TIMEOUT_SECONDS,
)
ontology_item = client.resolve_ontology(workspace_id, ONTOLOGY_DISPLAY_NAME)
original_definition = client.get_definition(
    workspace_id,
    ontology_item["id"],
)
metadata_plan = plan_metadata_update(
    original_definition,
    ONTOLOGY_METADATA,
)

print(f"Workspace: {workspace_name}")
print(f"Ontology: {ONTOLOGY_DISPLAY_NAME}")
for line in render_change_preview(metadata_plan):
    print(line)

## Apply and verify

After reviewing the preview, set `APPLY_CHANGES = True` and
`APPLY_CONFIRMATION = "APPLY ONTOLOGY METADATA"` in the parameter cell, then
set `EXCLUSIVE_APPLY_WINDOW_CONFIRMED = True` and run the Notebook again.
Writes are allowed only in an interactive run.

Before writing, the Notebook re-fetches the definition and aborts if anything
changed after the preview. After the LRO succeeds, it re-fetches and verifies
all 98 metadata objects and confirms that the structural digest is unchanged.
The Fabric API has no documented conditional-update precondition for this
operation, so close Ontology editors and pause all other writers for the apply
window before confirming the exclusive-write gate.

In [ ]:
if not APPLY_CHANGES:
    print("DRY_RUN_COMPLETE: no Fabric definition write was performed.")
else:
    if APPLY_CONFIRMATION != CONFIRMATION_PHRASE:
        raise OntologyMetadataError(
            f"Set APPLY_CONFIRMATION exactly to {CONFIRMATION_PHRASE!r}."
        )
    if not EXCLUSIVE_APPLY_WINDOW_CONFIRMED:
        raise OntologyMetadataError(
            "Close Ontology editors, pause every other writer, then set "
            "EXCLUSIVE_APPLY_WINDOW_CONFIRMED=True."
        )
    if not context.get("isForInteractive", False):
        raise OntologyMetadataError(
            "Metadata writes are allowed only in an interactive Notebook run."
        )

    latest_definition = client.get_definition(
        workspace_id,
        ontology_item["id"],
    )
    if definition_digest(latest_definition) != metadata_plan.original_digest:
        raise OntologyMetadataError(
            "The Ontology changed after preview. Re-run from the first cell."
        )

    latest_plan = plan_metadata_update(
        latest_definition,
        ONTOLOGY_METADATA,
    )
    if latest_plan.changes:
        client.update_definition(
            workspace_id,
            ontology_item["id"],
            latest_plan.definition,
        )
        write_status = "UPDATED"
    else:
        write_status = "ALREADY_CURRENT"

    verified_definition = client.get_definition(
        workspace_id,
        ontology_item["id"],
    )
    verification = verify_applied_metadata(
        verified_definition,
        ONTOLOGY_METADATA,
        expected_structural_digest=latest_plan.structural_digest,
    )
    print(
        "METADATA_APPLIED: "
        f"status={write_status}; "
        f"entities={verification['counts']['entityTypes']}; "
        f"staticProperties={verification['counts']['staticProperties']}; "
        f"timeseriesProperties={verification['counts']['timeseriesProperties']}; "
        f"relationships={verification['counts']['relationshipTypes']}; "
        f"manifestSha256={verification['manifestDigest']}"
    )